### Support Vector Regressor Implementation

In [1]:
#importing 'tips' dataset

import seaborn as sns
import numpy as np
import pandas as pd

dataset = sns.load_dataset('tips')
dataset.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [2]:
#Encoding categorical features

dataset['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [3]:
dataset['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [4]:
dataset['day'].value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

In [5]:
dataset['smoker'].value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [6]:
dataset['smoker'] = np.where(dataset['smoker']=='No', 0, 1)
dataset['smoker'].value_counts()

smoker
0    151
1     93
Name: count, dtype: int64

In [7]:
dataset['sex'] = np.where(dataset['sex']=='Female', 0, 1)
dataset['sex'].value_counts()

sex
1    157
0     87
Name: count, dtype: int64

In [8]:
dataset['time'] = np.where(dataset['time']=='Dinner', 0, 1)
dataset['time'].value_counts()

time
0    176
1     68
Name: count, dtype: int64

In [9]:
#applying onehot encoding on feature 'day'

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
day_data = encoder.fit_transform(dataset[['day']])
day_data

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 244 stored elements and shape (244, 4)>

In [10]:
df = pd.concat([dataset, pd.DataFrame(data=day_data.toarray(),columns=encoder.get_feature_names_out())], axis=1)
df.head()

,total_bill,tip,sex,smoker,day,time,size,day_Fri,day_Sat,day_Sun,day_Thur
0,16.99,1.01,0,0,Sun,0,2,0.0,0.0,1.0,0.0
1,10.34,1.66,1,0,Sun,0,3,0.0,0.0,1.0,0.0
2,21.01,3.50,1,0,Sun,0,3,0.0,0.0,1.0,0.0
3,23.68,3.31,1,0,Sun,0,2,0.0,0.0,1.0,0.0
4,24.59,3.61,0,0,Sun,0,4,0.0,0.0,1.0,0.0


In [11]:
df = df.drop('day', axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    int64  
 3   smoker      244 non-null    int64  
 4   time        244 non-null    int64  
 5   size        244 non-null    int64  
 6   day_Fri     244 non-null    float64
 7   day_Sat     244 non-null    float64
 8   day_Sun     244 non-null    float64
 9   day_Thur    244 non-null    float64
dtypes: float64(6), int64(4)
memory usage: 19.2 KB


In [12]:
#splitting data into train and test sets

x = df.drop('total_bill', axis=1)
y = df['total_bill']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [13]:
#model training

from sklearn.svm import SVR

svr = SVR()

svr.fit(x_train, y_train)
y_pred = svr.predict(x_test)

In [14]:
#performance metrics

from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r_value = r2_score(y_test, y_pred)

print(f"MAE: {mae}\nR_Squared: {r_value}")

MAE: 4.400967658119678
R_Squared: 0.5513989822828735


In [17]:
#hyperparameter tuning using gridsearchcv

from sklearn.model_selection import GridSearchCV

estimator = SVR()
params = dict(C=[1, 10, 0.1, 0.01],
              kernel=['rbf', 'linear', 'poly', 'sigmoid'],
              gamma=[0.1, 0.01, 0.001])

gridcv = GridSearchCV(estimator=estimator, param_grid=params, cv=5)

In [18]:
gridcv.fit(x_train, y_train)
gridcv.best_score_

np.float64(0.5041603090555462)

In [19]:
gridcv.best_params_

{'C': 10, 'gamma': 0.1, 'kernel': 'linear'}

In [20]:
#prediction

y_pred = gridcv.predict(x_test)

In [21]:
#performance metrics 


mae = mean_absolute_error(y_test, y_pred)
r_value = r2_score(y_test, y_pred)

print(f"MAE: {mae}\nR_Squared: {r_value}")


MAE: 4.050729501671692
R_Squared: 0.6598183597560396
